In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Pizza reviews.csv')

fake_col = "Are there ways for you to generate more data? Spliting up sentences, would that help?"
if fake_col in df.columns:
    df = df.drop(columns=[fake_col])

df = df[df['Review'].str.lower() != 'review']

df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
# Drop rows where Score or Review is missing
df = df.dropna(subset=['Score', 'Review'])

df = df[df['Review'].str.strip().str.len() > 2]

print(df.info())
print(df['Language'].value_counts())

<class 'pandas.core.frame.DataFrame'>
Index: 897 entries, 0 to 899
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Review    897 non-null    object 
 1   Score     897 non-null    float64
 2   Language  897 non-null    object 
dtypes: float64(1), object(2)
memory usage: 28.0+ KB
None
Language
English    491
Malay      402
Guga         2
Chinese      1
Nippon       1
Name: count, dtype: int64


In [6]:
print(df.head(25))

                                               Review  Score  \
0                                         Disgusting.      1   
1                              Loved the sambal kick!     10   
2                              Loved the sambal kick!    0.1   
3                              Loved the sambal kick!      5   
4   Satay on pizza? Surprisingly worked. Tasty, un...      7   
5         Mayo was too sweet. Didn't enjoy it at all.      3   
6   Spicy sambal was overwhelming for me, but the ...      6   
7   Crust was soggy, and the satay sauce didn’t he...      4   
8   Super creative! Sambal slice was fiery and bol...      7   
9   The sambal slice burned my mouth in the best w...      8   
10                                        Just weird.      2   
11  Incredible mix of flavors. Satay on pizza is a...      9   
12                Sambal overwhelmed everything else.      4   
13  Satay chicken worked well on pizza. Creative b...      6   
14      Enjoyed every bite, especially t

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# --- 1. Target Engineering (Sentiment Classification) ---
def assign_sentiment_class(score):
    if score <= 4:
        return 0  # Negative
    elif score <= 6:
        return 1  # Neutral
    else:
        return 2  # Positive

df['Label'] = df['Score'].apply(assign_sentiment_class)

# --- 2. Tokenization & Sequence Padding ---
MAX_WORDS = 5000       # Vocabulary size limit
MAX_SEQUENCE_LENGTH = 60  # Length constraint per review

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['Review'].astype(str))

sequences = tokenizer.texts_to_sequences(df['Review'].astype(str))
X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

# --- 3. Label Processing ---
# One-hot encode the target labels for multi-class classification
y = pd.get_dummies(df['Label']).values

# --- 4. Train-Validation Split ---
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training shape: {X_train.shape}, Validation shape: {X_val.shape}")

c:\Users\JunHao\anaconda3\envs\gpu_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Training shape: (717, 60), Validation shape: (180, 60)


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

EMBEDDING_DIM = 64

# Building Vanilla SimpleRNN Network
baseline_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM),
    SimpleRNN(units=32),
    Dense(units=3, activation='softmax')  # 3 Units matching Negative, Neutral, Positive
])

baseline_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

baseline_model.summary()

# Training Baseline
history_baseline = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 64)          320000    
                                                                 
 simple_rnn (SimpleRNN)      (None, 32)                3104      
                                                                 
 dense (Dense)               (None, 3)                 99        
                                                                 
Total params: 323,203
Trainable params: 323,203
Non-trainable params: 0
_________________________________________________________________
Epoch 1/10
23/23 [==============================] - 1s 20ms/step - loss: 0.9280 - accuracy: 0.5969 - val_loss: 0.8039 - val_accuracy: 0.6556
Epoch 2/10
23/23 [==============================] - 0s 11ms/step - loss: 0.6821 - accuracy: 0.7071 - val_loss: 0.6971 - val_accuracy: 0.6833
Epoch 3/10
23/23 [==============

In [12]:
from tensorflow.keras.layers import LSTM, Dropout, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint

# --- Optimization Layer Architecture ---
improved_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM),
    # Bidirectional LSTM handles long-term contexts in sequences better than Vanilla RNN
    Bidirectional(LSTM(units=64, return_sequences=False)),
    Dropout(0.5), # Regularization mechanism to counter strict dataset sizing
    Dense(units=3, activation='softmax')
])

improved_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Mandatory checkpoint configuration to output your '.h5' file containing optimal weights
checkpoint_callback = ModelCheckpoint(
    filepath='best_neural_network_weights.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Training Optimized Model
history_improved = improved_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[checkpoint_callback]
)

Epoch 1/15
21/23 [==========================>...] - ETA: 0s - loss: 1.0758 - accuracy: 0.4137
Epoch 1: val_accuracy improved from -inf to 0.62778, saving model to best_neural_network_weights.h5
23/23 [==============================] - 4s 61ms/step - loss: 1.0727 - accuracy: 0.4212 - val_loss: 1.0309 - val_accuracy: 0.6278
Epoch 2/15
22/23 [===========================>..] - ETA: 0s - loss: 0.9708 - accuracy: 0.5866
Epoch 2: val_accuracy improved from 0.62778 to 0.68333, saving model to best_neural_network_weights.h5
23/23 [==============================] - 1s 32ms/step - loss: 0.9726 - accuracy: 0.5858 - val_loss: 0.8366 - val_accuracy: 0.6833
Epoch 3/15
23/23 [==============================] - ETA: 0s - loss: 0.7136 - accuracy: 0.7155
Epoch 3: val_accuracy improved from 0.68333 to 0.71667, saving model to best_neural_network_weights.h5
23/23 [==============================] - 1s 32ms/step - loss: 0.7136 - accuracy: 0.7155 - val_loss: 0.6452 - val_accuracy: 0.7167
Epoch 4/15
22/23 [====